# explainrec demo

End-to-end walkthrough of the LLM-explained recommender:
estimate ratings -> solve the allocation LP -> ask natural-language
what-if questions -> the LLM edits the problem, we re-solve, compare,
and the LLM explains the comparison.

LLM calls go through the **local Claude Code CLI** (subscription auth,
no API key). Switch to `ApiBackend()` if you prefer the API.


In [1]:
# make the repo root importable regardless of where Jupyter was launched
import sys
from pathlib import Path
try:
    import explainrec  # noqa: F401
except ModuleNotFoundError:
    root = next(p for p in Path.cwd().resolve().parents
                if (p / "explainrec").is_dir())
    sys.path.insert(0, str(root))

from explainrec.pipeline import Pipeline
from explainrec.llm.backend import CliBackend
from explainrec.compare import report_text

backend = CliBackend(model="opus")   # or ApiBackend() with ANTHROPIC_API_KEY
p = Pipeline.build()                 # downloads ML-100k on first run, fits the rating model
data = p.baseline.data
print(f"train RMSE: {p.baseline.model.train_rmse:.3f}")
print(data.summary())

train RMSE: 0.832
943 users, 1682 movies, 100000 ratings (1-5 stars). Cold items: the 755 movies with at most 20 ratings. User attributes: age, gender (M/F), occupation. Genres: unknown, Action, Adventure, Animation, Children's, Comedy, Crime, Documentary, Drama, Fantasy, Film-Noir, Horror, Musical, Mystery, Romance, Sci-Fi, Thriller, War, Western.


## Baseline
Total-rating maximization with one fairness constraint: every cold item
must reach at least 5 users. Solving the full LP takes ~40 s; the solution
is cached on the pipeline afterwards.


In [2]:
sol = p.base_solution
cold = data.cold_items
print(f"objective: {sol.objective:.1f}   mean predicted rating: {sol.mean_predicted_rating:.3f}")
print(f"cold items shown: {int((sol.exposure[cold] > 1e-6).sum())}/{len(cold)}")
print(f"solve time: {sol.solve_seconds:.1f}s   fractional entries: {sol.n_fractional}")
print("\nslate of user 42:")
for i in sol.recs[42]:
    print(f"  {data.title(i)}")

objective: 38481.1   mean predicted rating: 4.081
cold items shown: 755/755
solve time: 43.7s   fractional entries: 0

slate of user 42:
  Madonna: Truth or Dare (1991)
  Ayn Rand: A Sense of Life (1997)
  Braveheart (1995)
  Police Story 4: Project S (Chao ji ji hua) (1993)
  Shawshank Redemption, The (1994)
  Boys of St. Vincent, The (1993)
  Lassie (1994)
  Titanic (1997)
  Schindler's List (1993)
  Good Will Hunting (1997)


## What-if 1: stop exploring
A content-strategy question: what does the cold-start promotion cost us?


In [3]:
r1 = p.ask("What happens if we stop promoting cold items?", backend=backend)
print("modification:", r1.modification.summary)
print()
print(report_text(r1.report))

modification: Remove the cold-item exposure constraint so cold items are no longer guaranteed any exposure.

objective:
  base: 38481.1
  modified: 42367.3
  delta_pct: 10.1
mean_predicted_rating:
  base: 4.081
  modified: 4.493
users_with_changed_slate:
  count: 782
  fraction: 0.829
cold_item_exposure:
  base_total: 3775.0
  modified_total: 24.0
  base_items_shown: 755
  modified_items_shown: 7
  n_cold_items: 755
  mean_cold_items_per_user_slate:
    base: 4.0
    modified: 0.03
per_user_slate_rating_distribution:
  note: mean predicted rating of each user's slate; min = worst-off user
  base:
    min: 1.403
    p25: 3.807
    median: 4.109
    p75: 4.386
    max: 5.0
    std: 0.432
  modified:
    min: 2.848
    p25: 4.292
    median: 4.512
    p75: 4.72
    max: 5.0
    std: 0.316
exploration_burden_distribution:
  note: number of cold items in each user's slate; gini 0 = evenly shared
  base:
    min: 0.0
    p25: 1.0
    median: 4.0
    p75: 6.0
    max: 10.0
    std: 2.886
    

In [4]:
print(r1.explanation)

Dropping the cold-item guarantee makes the platform's efficiency objective jump about 10.1% (38,481 to 42,367), with mean predicted rating rising from 4.08 to 4.49. That gain is real but it comes almost entirely at the expense of new items: cold-item exposure collapses from 3,775 slots across 755 items to just 24 slots across 7 items, so effectively the entire cold catalog goes dark. Average cold items per slate falls from 4.0 to 0.03.

Users benefit fairly broadly. 83% of slates change, and even the worst-served user improves, with the minimum slate rating going from 1.40 to 2.85 and the spread across users tightening (std 0.43 to 0.32). The exploration burden that was previously shared, if unevenly (Gini 0.41), now falls on a tiny handful of users (Gini 0.99).

The cost lands on item providers and on long-run catalog health. Exposure concentration rises sharply: the catalog Gini goes from 0.788 to 0.988, and the top 10% of items capture 100% of exposure, versus 66% before. New titles

## What-if 2: individual gender counterfactual
A user-facing recourse question. Note the *non-locality*: because the
exposure constraints couple users, flipping one user's gender also
perturbs other users' slates.


In [5]:
r2 = p.ask("Would user 42 get the same movies if she were male?", backend=backend)
print("modification:", r2.modification.summary)
print()
print(report_text(r2.report))

modification: Re-solve with user 42's gender counterfactually set to male, keeping all other settings unchanged, to compare their slate against the baseline.

objective:
  base: 38481.1
  modified: 38480.8
  delta_pct: -0.0
mean_predicted_rating:
  base: 4.081
  modified: 4.081
users_with_changed_slate:
  count: 54
  fraction: 0.057
cold_item_exposure:
  base_total: 3775.0
  modified_total: 3775.0
  base_items_shown: 755
  modified_items_shown: 755
  n_cold_items: 755
  mean_cold_items_per_user_slate:
    base: 4.0
    modified: 4.0
per_user_slate_rating_distribution:
  note: mean predicted rating of each user's slate; min = worst-off user
  base:
    min: 1.403
    p25: 3.807
    median: 4.109
    p75: 4.386
    max: 5.0
    std: 0.432
  modified:
    min: 1.403
    p25: 3.807
    median: 4.107
    p75: 4.387
    max: 5.0
    std: 0.432
exploration_burden_distribution:
  note: number of cold items in each user's slate; gini 0 = evenly shared
  base:
    min: 0.0
    p25: 1.0
    media

In [6]:
print(r2.explanation)

No, she would not. Flipping user 42's gender to male changes half her slate: 5 of her 10 titles stay (including *Braveheart*, *Schindler's List*, *Titanic*), and 5 are swapped out for others such as *Casablanca*, *Raiders of the Lost Ark*, and *Close Shave*.

Two differences stand out. Her slate's mean predicted rating rises from 4.091 to 4.459, and her share of cold (rarely-shown) items drops from 5 to 2 out of 10. So in the male version she is served more mainstream, higher-confidence titles and carries less of the platform's exploration burden, which other users then absorb.

At the system level almost nothing moves. The total predicted rating (the platform's efficiency objective) shifts from 38,481.1 to 38,480.8, essentially zero, mean predicted rating stays at 4.081, and total cold-item exposure is unchanged at 3,775 across the same 755 items. Item-exposure concentration is identical (Gini 0.788). Interestingly, 54 users (5.7%) see a changed slate, not just user 42, because the sh

## Bypassing the LLM
For experiments, inject a `Modification` directly - the deterministic
core (`run_modification`) is separate from the LLM layer.


In [7]:
from explainrec.scenario import Modification
from explainrec.constraints import ConstraintSpec, ItemSelector

mod = Modification(
    summary="relax cold exposure from 5 to 2 users per item",
    remove_constraints=["cold-item-exposure"],
    add_constraints=[ConstraintSpec(
        name="cold-item-exposure-relaxed", type="min_item_exposure",
        items=ItemSelector(kind="cold"), min_users=2,
    )],
)
print(report_text(p.run_modification(mod)))

objective:
  base: 38481.1
  modified: 40902.1
  delta_pct: 6.29
mean_predicted_rating:
  base: 4.081
  modified: 4.337
users_with_changed_slate:
  count: 780
  fraction: 0.827
cold_item_exposure:
  base_total: 3775.0
  modified_total: 1510.0
  base_items_shown: 755
  modified_items_shown: 755
  n_cold_items: 755
  mean_cold_items_per_user_slate:
    base: 4.0
    modified: 1.6
per_user_slate_rating_distribution:
  note: mean predicted rating of each user's slate; min = worst-off user
  base:
    min: 1.403
    p25: 3.807
    median: 4.109
    p75: 4.386
    max: 5.0
    std: 0.432
  modified:
    min: 1.592
    p25: 4.128
    median: 4.382
    p75: 4.602
    max: 5.0
    std: 0.372
exploration_burden_distribution:
  note: number of cold items in each user's slate; gini 0 = evenly shared
  base:
    min: 0.0
    p25: 1.0
    median: 4.0
    p75: 6.0
    max: 10.0
    std: 2.886
    gini: 0.409
  modified:
    min: 0.0
    p25: 0.0
    median: 0.0
    p75: 2.0
    max: 10.0
    std: 2.5

Further reading: `README.md` (usage), `docs/architecture.md`
(formulation, modification schema, extension guide), and the web demo
(`python demo/server.py`).
